In [8]:
from typing import Optional
from pathlib import Path

In [9]:
import pandas as pd

csv_path = Path("/home/hmx42/Projects/Kronos_projest/finetune_csv/data/HK_ali_09988_kline_5min_all.csv")

df = pd.read_csv(csv_path)
df["timestamps"] = pd.to_datetime(df["timestamps"])
df = df.sort_values("timestamps").reset_index(drop=True)

df.head()

,timestamps,open,close,high,low,volume,amount
0,2019-11-26 09:35:00,182.45215,184.45215,184.95215,182.45215,15136000,0
1,2019-11-26 09:40:00,184.35215,183.85215,184.55215,183.45215,4433300,0
2,2019-11-26 09:45:00,183.85215,183.35215,183.95215,182.95215,3070900,0
3,2019-11-26 09:50:00,183.35215,183.75215,183.95215,183.25215,2288600,0
4,2019-11-26 09:55:00,183.85215,183.85215,184.15215,183.75215,1770000,0


In [10]:
lookback = 400
pred_len = 120

In [11]:
x_df = df.loc[:lookback-1, ['open', 'high', 'low', 'close', 'volume', 'amount']]
x_timestamp = df.loc[:lookback-1, 'timestamps']
y_timestamp = df.loc[lookback:lookback+pred_len-1, 'timestamps']

In [12]:
x_df.shape, x_timestamp.shape, y_timestamp.shape

((400, 6), (400,), (120,))

In [13]:
def calc_time_stamps(timestamps_series):
    time_df=pd.DataFrame()
    time_df["minute"]=timestamps_series.dt.minute
    time_df["hour"]=timestamps_series.dt.hour
    time_df["day"]=timestamps_series.dt.day
    time_df["month"]=timestamps_series.dt.month
    time_df["year"]=timestamps_series.dt.year
    time_df["weekday"]=timestamps_series.dt.weekday
    return time_df

In [ ]:
x_time_df = calc_time_stamps(x_timestamp)
y_time_df = calc_time_stamps(y_timestamp)

x_time_df.head(), y_time_df.head()


(   minute  hour  day  month  year  weekday
 0      35     9   26     11  2019        1
 1      40     9   26     11  2019        1
 2      45     9   26     11  2019        1
 3      50     9   26     11  2019        1
 4      55     9   26     11  2019        1,
      minute  hour  day  month  year  weekday
 400      55     9    4     12  2019        2
 401       0    10    4     12  2019        2
 402       5    10    4     12  2019        2
 403      10    10    4     12  2019        2
 404      15    10    4     12  2019        2)

In [16]:
import numpy as np
feature_cols=["open", "high", "low", "close", "volume", "amount"]

x=x_df[feature_cols].values.astype(np.float32)

In [18]:
x_mean = np.mean(x, axis=0)
x_std = np.std(x, axis=0)

clip = 5
x_norm = (x - x_mean) / (x_std + 1e-5)
x_norm = np.clip(x_norm, -clip, clip)

x_norm.shape, x_mean, x_std

((400, 6),
 array([1.9047766e+02, 1.9068039e+02, 1.9025891e+02, 1.9048090e+02,
        5.9411425e+05, 0.0000000e+00], dtype=float32),
 array([4.3197227e+00, 4.3538923e+00, 4.2665062e+00, 4.3052406e+00,
        9.2559112e+05, 0.0000000e+00], dtype=float32))